# VANDAlize — 01: Activation Extraction

Extracts last-token residual-stream activations from a causal LM for a set of
harmful/benign prompts, across all layers, and saves them for analysis.

**You need a GPU runtime** (free Colab T4 is enough) and access to the gated
model weights on HuggingFace. Loading a 13B model in 4-bit takes ~15-20 min on a T4.

Run this once per (model, condition) pair. Conditions: `english`, `filipino`,
`taglish`. Models in the paper: `meta-llama/Llama-2-13b-hf`,
`aisingapore/Llama-SEA-LION-v2-...` (SEA-LION v1), `aisingapore/Llama-SEA-LION-v3-8B`.

**Most reviewers do not need to run this notebook** — the saved activations and
the analysis notebook (`02_analysis.ipynb`) are sufficient to reproduce every
figure. This notebook is provided for full reproducibility from raw weights.

Convention: in the dataset CSV, column `label` uses `1` = harmful, `0` = benign.

## Setup

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes torch numpy pandas

In [ ]:
import torch
import numpy as np
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from tqdm.auto import tqdm

# --- Provide your HuggingFace token via environment, do NOT hardcode it ---
import os, getpass
HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("HuggingFace token: ")
assert HF_TOKEN, "A HuggingFace token with access to the gated model is required."

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
assert DEVICE == "cuda", "A GPU runtime is required for extraction."
print("Device:", DEVICE)

## Choose model and condition

Set `MODEL_NAME` and the input CSV for the condition you are extracting.
The CSV must have columns `statements` (text) and `label` (0 harmful / 1 benign).

In [ ]:
# --- EDIT THESE TWO LINES PER RUN ---
MODEL_NAME = "meta-llama/Llama-2-13b-hf"   # or the SEA-LION v2 / v3 model id
CONDITION  = "english"                       # "english" | "filipino" | "taglish"
RUN        = 1                               # run index, if extracting multiple runs

CSV_PATH   = f"data/{CONDITION}_{RUN}.csv"   # statements,label
OUT_STEM   = f"{CONDITION}_{RUN}"            # output filename stem

## Load model (4-bit quantized)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto", token=HF_TOKEN,
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

NUM_LAYERS = len(model.model.layers)
D_MODEL = model.config.hidden_size
print(f"{MODEL_NAME}: {NUM_LAYERS} layers, hidden dim {D_MODEL}")

## Extraction function

In [ ]:
@torch.no_grad()
def extract_activations(statements, model, tokenizer, layers, batch_size=25):
    """Last-token hidden-state activations at each requested layer.

    Returns dict {layer_idx: tensor [n_statements, hidden]} on CPU, float32.
    Uses output_hidden_states; hidden_states[L] is the residual stream after layer L
    (index 0 is the embedding output, so layer index L maps to hidden_states[L+1]).
    """
    out = {L: [] for L in layers}
    for start in tqdm(range(0, len(statements), batch_size)):
        batch = statements[start:start + batch_size]
        enc = tokenizer(batch, return_tensors="pt", padding=True, truncation=True).to(model.device)
        result = model(**enc, output_hidden_states=True)
        # index of the last non-pad token for each sequence
        last_idx = enc["attention_mask"].sum(dim=1) - 1
        for L in layers:
            hs = result.hidden_states[L + 1]            # [batch, seq, hidden]
            picked = hs[torch.arange(hs.size(0)), last_idx]  # [batch, hidden]
            out[L].append(picked.float().cpu())
    return {L: torch.cat(v, dim=0) for L, v in out.items()}

## Run extraction and save

In [ ]:
df = pd.read_csv(CSV_PATH)
statements = df["statements"].tolist()
labels = torch.tensor(df["label"].values, dtype=torch.float32)
print(f"{CONDITION}: n={len(statements)}, harmful={(labels==0).sum().item()}, benign={(labels==1).sum().item()}")

acts = extract_activations(statements, model, tokenizer, list(range(NUM_LAYERS)))

# Wrap with a named key so the analysis notebook can find it
key = f"{CONDITION} run {RUN}"
torch.save({key: acts},   f"{OUT_STEM}_activations_all_layers.pt")
torch.save({key: labels}, f"{OUT_STEM}_labels.pt")
print("Saved:", f"{OUT_STEM}_activations_all_layers.pt", "and", f"{OUT_STEM}_labels.pt")

---
Repeat for every (model, condition) pair. Then open **`02_analysis.ipynb`**
to reproduce all figures and tables.